In [6]:
import random
import numpy as np
import time
from IPython.display import clear_output

class SnakeEnv:
    def __init__(self, width=10, height=10):
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        # Start in the middle moving right
        self.snake = [[self.height // 2, self.width // 2], 
                      [self.height // 2, (self.width // 2) - 1]]
        self.direction = [0, 1] # [row_change, col_change] -> Right
        self.food = self._place_food()
        self.score = 0
        self.game_over = False
        return self.get_state()

    def _place_food(self):
        while True:
            food = [random.randint(0, self.height - 1), random.randint(0, self.width - 1)]
            if food not in self.snake:
                return food

    def get_state(self):
        head = self.snake[0]
        
        # Define directions relative to global space
        dirs = [[-1, 0], [1, 0], [0, -1], [0, 1]] # Up, Down, Left, Right
        
        # Calculate straight, right, left shifts based on current heading
        r, c = self.direction
        dir_idx = dirs.index(self.direction)
        
        # Straight, Right, Left relative directions
        rel_dirs = [
            self.direction,                 # Straight
            [c, -r],                        # Right turn
            [-c, r]                         # Left turn
        ]
        
        # 1. Danger sensing
        danger = []
        for rd in rel_dirs:
            next_step = [head[0] + rd[0], head[1] + rd[1]]
            if (next_step[0] < 0 or next_step[0] >= self.height or 
                next_step[1] < 0 or next_step[1] >= self.width or 
                next_step in self.snake):
                danger.append(1)
            else:
                danger.append(0)
                
        # 2. Current direction state
        current_dir = [1 if i == dir_idx else 0 for i in range(4)]
        
        # 3. Food direction state
        food_dir = [
            1 if self.food[0] < head[0] else 0, # Food Up
            1 if self.food[0] > head[0] else 0, # Food Down
            1 if self.food[1] < head[1] else 0, # Food Left
            1 if self.food[1] > head[1] else 0  # Food Right
        ]
        
        return tuple(danger + current_dir + food_dir)

    def step(self, action):
        # Actions: 0 = Straight, 1 = Right Turn, 2 = Left Turn
        dirs = [[-1, 0], [1, 0], [0, -1], [0, 1]] # Up, Down, Left, Right
        r, c = self.direction
        
        if action == 1:    # Right turn
            self.direction = [c, -r]
        elif action == 2:  # Left turn
            self.direction = [-c, r]
            
        head = self.snake[0]
        new_head = [head[0] + self.direction[0], head[1] + self.direction[1]]
        
        # Check collision
        if (new_head[0] < 0 or new_head[0] >= self.height or 
            new_head[1] < 0 or new_head[1] >= self.width or 
            new_head in self.snake):
            self.game_over = True
            return self.get_state(), -10, True # Penalty for dying
            
        self.snake.insert(0, new_head)
        
        # Check if eating food
        if new_head == self.food:
            self.score += 1
            self.food = self._place_food()
            reward = 10 # Big reward for food
        else:
            self.snake.pop()
            reward = 0 # Neutral step
            
        return self.get_state(), reward, False

    def render(self):
        board = [["." for _ in range(self.width)] for _ in range(self.height)]
        board[self.food[0]][self.food[1]] = "F"
        for idx, pt in enumerate(self.snake):
            board[pt[0]][pt[1]] = "O" if idx == 0 else "o"
            
        out = f"Score: {self.score}\n"
        out += "\n".join([" ".join(row) for row in board])
        print(out)

class QAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0, min_epsilon=0.01, decay=0.995):
        self.q_table = {}
        self.alpha = alpha       # Learning rate
        self.gamma = gamma       # Discount factor
        self.epsilon = epsilon   # Exploration rate
        self.min_epsilon = min_epsilon
        self.decay = decay

    def get_q(self, state):
        if state not in self.q_table:
            self.q_table[state] = np.zeros(3) # 3 actions
        return self.q_table[state]

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 2)
        return np.argmax(self.get_q(state))

    def learn(self, state, action, reward, next_state, done):
        old_q = self.get_q(state)[action]
        next_max = 0 if done else np.max(self.get_q(next_state))
        
        # Q-learning formula
        self.q_table[state][action] = old_q + self.alpha * (reward + self.gamma * next_max - old_q)
        
        if done and self.epsilon > self.min_epsilon:
            self.epsilon *= self.decay

In [7]:
env = SnakeEnv(width=10, height=10)
agent = QAgent()

episodes = 2000
print("Training agent...")

for ep in range(episodes):
    state = env.reset()
    done = False
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        state = next_state

print(f"Training finished over {episodes} rounds! Q-table size: {len(agent.q_table)} states found.")

Training agent...
Training finished over 2000 rounds! Q-table size: 256 states found.


In [8]:
# Turn off exploration for the test drive
agent.epsilon = 0.0

state = env.reset()
done = False

try:
    while not done:
        clear_output(wait=True)
        env.render()
        action = agent.choose_action(state)
        state, _, done = env.step(action)
        time.sleep(0.15) # Controls speed
    
    clear_output(wait=True)
    env.render()
    print("\nGame Over!")
except KeyboardInterrupt:
    print("\nPlayback stopped early.")

Score: 26
o o o o o o o o . .
o o o o o o o o . .
o o . . . . . . . .
o o . . . . . . . .
O o . . . . . . . .
o o F . . . . . . .
o . . . . . . . . .
o . . . . . . . . .
o o . . . . . . . .
. . . . . . . . . .

Game Over!
